In [1]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts

In [2]:
import re
import pandas as pd
import logging

from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import call_api_with_accountId, send_to_innkeepr_api_paginated

In [3]:
workspace_ids = return_workspace_ids()
url = return_api_url()

In [4]:
path = "/Users/karolinegriesbach/Downloads/message-4.txt"
with open(path, encoding="utf-8") as f:
    raw = f.read()

header_re = re.compile(r"^(?P<customer>.+?)\s*\(\d+\):\s*$")
item_re = re.compile(r"^\s*○\s*(?P<audience>.+?)\s*\|\s*(?P<id>\S+)\s*$")


rows = []
current_customer = None
for line in raw.splitlines():
    if not line.strip():
        continue
    h = header_re.match(line)
    if h:
        current_customer = h.group("customer").strip()
        continue
    m = item_re.match(line)
    if m:
        rows.append({
            "audience": m.group("audience"),
            "id": m.group("id"),
            "workspace": current_customer,
        })

missing_audiences = pd.DataFrame(rows, columns=["audience", "id", "workspace"])
missing_audiences

In [5]:
missing_audiences["workspace_id"] = missing_audiences["workspace"].apply(lambda x: [acc["id"] for acc in workspace_ids if acc["name"] == x][0])
missing_audiences

In [6]:
result_with_treatment_check = pd.DataFrame(columns=["workspace","audience", "audience_id", "connectionId","externalId"])
for workspace_id in missing_audiences["workspace_id"].unique():
    workspace_df = missing_audiences[missing_audiences["workspace_id"] == workspace_id]
    audience_usage = call_api_with_accountId(
        endpoint_url=f"{url}/audiences/usage/query",
        accountID=workspace_id,
        content={"fromDate": "20260426","toDate": "20260429"},
        logger=logging
    )
    audience_usage = pd.json_normalize(audience_usage)
    for row_workspace in workspace_df.iterrows():
        workspace = row_workspace[1]["workspace"]
        audience_name = row_workspace[1]["audience"]
        audience_id = row_workspace[1]["id"]
        source = call_api_with_accountId(
            endpoint_url=f"{url}/audiences/query",
            accountID=workspace_id,
            content={"id":audience_id},
            logger=logging
        )
        if len(source)!= 1:
            raise ("Expected only one source")
        source = [entry["connection"] for entry in source][0]
        print(f"{workspace} - audience: {audience_name}")
        temp_usage = audience_usage[audience_usage["targetings"].astype("str").str.contains(audience_name).fillna(False)]
        temp_usage = temp_usage[temp_usage["connectionId"]==source]
        if temp_usage.empty:
            result_with_treatment_check.loc[len(result_with_treatment_check)] = [workspace, audience_name, audience_id, None, None]
            continue
        external_ids = set(temp_usage["treatment"].unique().tolist())
        source = set(temp_usage["connectionId"].unique().tolist())
        result_with_treatment_check.loc[len(result_with_treatment_check)] = [workspace, audience_name, audience_id, source, external_ids]
result_with_treatment_check

In [8]:
result_with_treatment_check.to_csv("SprintStories/Missing-DTW-Audiences/unused_audiences.csv", index=False)